# Feature Engineering

1. IMPORT PACKAGES

In [3]:
import pandas as pd
df = pd.read_csv('/content/clean_data_after_eda.csv')
display(df.head(3))
print(df.shape)
print(df["date_activ"].dtype)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


(14606, 44)
object


2. LOAD DATA

In [4]:
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')
display(df.head(3))
print(df["date_activ"].dtype)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


datetime64[ns]


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [5]:
price_df = pd.read_csv('/content/price_data (1).csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')

# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

## Step 1 — Remove unnecessary columns

In [6]:
# Check for columns with only one unique value
constant_columns = [
    col for col in df.columns
    if df[col].nunique() <= 1
]

print("Constant columns:", constant_columns)

Constant columns: []


So we don't remove anything at this stage.

## Step 2 — Expand the dataset by extracting useful information from the date columns

- date_activ
- date_end
- date_modif_prod
- date_renewal

The idea is to extract useful information from these dates instead of giving the raw dates directly to the model.

In [7]:
#Create customer tenure in months
df['tenure_days'] = (df['date_end'] - df['date_activ']).dt.days
df['tenure_months'] = df['tenure_days'] / 30

#Create days until renewal
df['days_to_renewal'] = (df['date_renewal'] - df['date_activ']).dt.days

#Extract month from activation date
df['activation_month'] = df['date_activ'].dt.month

#Extract month from product modification
df['modification_month'] = df['date_modif_prod'].dt.month

#Check the new columns
df[['tenure_days',
    'tenure_months',
    'days_to_renewal',
    'activation_month',
    'modification_month']].head(5)

,tenure_days,tenure_months,days_to_renewal,activation_month,modification_month
0,1096,36.533333,738,6,11
1,2566,85.533333,2201,8,8
2,2192,73.066667,1827,4,4
3,2192,73.066667,1827,3,3
4,2245,74.833333,1881,1,1


## Step 3 — Create new features from existing columns

In [8]:
# Create total consumption
df['total_consumption'] = (
    df['cons_12m'] + df['cons_gas_12m']
)

# Create consumption change
df['consumption_change'] = (
    df['cons_last_month'] - (df['cons_12m'] / 12)
)

#Create average monthly consumption
df['avg_monthly_consumption'] = df['cons_12m'] / 12

#Check the newly created features
df[[
    'total_consumption',
    'consumption_change',
    'avg_monthly_consumption',
]].head(5)

,total_consumption,consumption_change,avg_monthly_consumption
0,54946,0.000000,0.000000
1,4660,-388.333333,388.333333
2,544,-45.333333,45.333333
3,1584,-132.000000,132.000000
4,4425,157.250000,368.750000


### Why are these features created?

- **`total_consumption`**
  - Combines electricity and gas consumption.
  - Gives an **overall measure of customer energy usage**.

- **`consumption_change`**
  - Compares recent consumption with average monthly consumption.
  - Helps identify **increases or decreases in customer usage**.

- **`avg_monthly_consumption`**
  - Converts annual consumption into a monthly average.
  - Makes **annual and monthly consumption easier to compare**.

## Finally - combine these datasets

In [9]:
df = pd.merge(
    df,
    price_df,
    on='id',
    how='left'
)
df.head(5)
print(df.shape)

(175149, 59)
